In [4]:
# ========================================================
# CELL 1: CONFIGURATION
# ========================================================
import os
import getpass
from pathlib import Path
from dotenv import load_dotenv

# Load .env file
load_dotenv()

# Thai PBS API
BASE_URL = "https://api.thaipbs.net"

# ThaiLLM OpenAI-compatible API
THAILLM_BASE_URL = "http://thaillm.or.th/api/v1"
THAILLM_MODEL = "openthaigpt-thaillm-8b-instruct-v7.2"

# Load 7 API keys from .env
THAILLM_API_KEYS = [
    os.getenv("THAILLM_API_KEY1"),
    os.getenv("THAILLM_API_KEY2"),
    os.getenv("THAILLM_API_KEY3"),
    os.getenv("THAILLM_API_KEY4"),
    os.getenv("THAILLM_API_KEY5"),
    os.getenv("THAILLM_API_KEY6"),
    os.getenv("THAILLM_API_KEY7"),
]

# Filter out None values
THAILLM_API_KEYS = [key for key in THAILLM_API_KEYS if key]

if not THAILLM_API_KEYS:
    THAILLM_API_KEY = getpass.getpass("Enter ThaiLLM API key: ")
    THAILLM_API_KEYS = [THAILLM_API_KEY]
else:
    print(f"Loaded {len(THAILLM_API_KEYS)} API keys from .env")

# Use first key for backward compatibility
THAILLM_API_KEY = THAILLM_API_KEYS[0]

# None = ข่าวย้อนหลังทั้งหมดที่ API เปิดให้เข้าถึง
START_DATE = None          # ตัวอย่าง "2024-01-01"
END_DATE = None            # ตัวอย่าง "2026-10-23"

# แหล่งข่าว Thai PBS ที่ต้องการรวบรวม
SITES = ["news", "now", "theactive", "policywatch", "verify"]

# API limits
PAGE_SIZE = 20
MAX_PAGES_PER_SITE = 10000
THAIPBS_DELAY = 0.15

# ThaiLLM Rate Limits: 5 req/s, 200 req/min PER API KEY
# Each API key has its own limit, so 7 keys = 35 req/s total
# delay = 1 / 5 = 0.2s, use 0.25s for safety margin
THAILLM_DELAY = 0.25

# จำกัดจำนวนข่าวสำหรับทดสอบครั้งแรก
# None = ไม่จำกัด
MAX_ARTICLES = None        # เช่น 20 เพื่อทดลอง pipeline

# Output directory
OUT_DIR = Path(
    r"D:\Project_Github\gold assisstant\dataset\news"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = OUT_DIR / "thaipbs_raw.jsonl"
CLASSIFIED_FILE = OUT_DIR / "thaipbs_classified.jsonl"
GOLD_FILE = OUT_DIR / "gold_related_news.jsonl"
ERROR_FILE = OUT_DIR / "collection_errors.jsonl"
CACHE_FILE = OUT_DIR / "thaipbs_cache.json"

print("Output directory:", OUT_DIR)
print("ThaiLLM model:", THAILLM_MODEL)
print("Date range:", START_DATE, "to", END_DATE)
print(f"Using {len(THAILLM_API_KEYS)} API keys for parallel processing")
print(f"Rate limit per key: 5 req/s (200 req/min)")
print(f"Total rate limit: ~{len(THAILLM_API_KEYS) * 5} req/s (7 keys × 5 req/s)")
print(f"Actual rate per worker: ~{1/THAILLM_DELAY:.2f} req/s")

Loaded 7 API keys from .env
Output directory: D:\Project_Github\gold assisstant\dataset\news
ThaiLLM model: openthaigpt-thaillm-8b-instruct-v7.2
Date range: None to None
Using 7 API keys for parallel processing
Rate limit per key: 5 req/s (200 req/min)
Total rate limit: ~35 req/s (7 keys × 5 req/s)
Actual rate per worker: ~4.00 req/s


In [ ]:
# ========================================================
# CELL 2: THAI PBS COLLECTOR
# ========================================================
import json
import time
import hashlib
import requests
import pandas as pd

from datetime import datetime, timezone
from urllib.parse import urljoin

session = requests.Session()
session.headers.update({
    "User-Agent": "GoldAssistant-NewsCollector/1.0",
    "Accept": "application/json",
})

# ---------- JSONL utilities ----------

def append_jsonl(path, obj):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def read_jsonl(path):
    if not path.exists():
        return []

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows


def load_json_cache(path):
    if not path.exists():
        return {}
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, OSError):
        return {}


def save_json_cache(path, cache):
    tmp = path.with_suffix(".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)
    os.replace(tmp, path)


detail_cache = load_json_cache(CACHE_FILE)


# ---------- API client ----------

def api_get(path, params=None, retries=4):
    url = urljoin(BASE_URL.rstrip("/") + "/", path.lstrip("/"))

    for attempt in range(retries):
        try:
            response = session.get(url, params=params, timeout=30)

            if response.status_code == 429:
                wait = min(2 ** (attempt + 1), 30)
                print(f"Rate limited by Thai PBS; waiting {wait}s")
                time.sleep(wait)
                continue

            response.raise_for_status()
            return response.json()

        except (requests.RequestException, ValueError) as exc:
            if attempt == retries - 1:
                append_jsonl(ERROR_FILE, {
                    "stage": "thaipbs_api",
                    "path": path,
                    "params": params,
                    "error": str(exc),
                    "timestamp": datetime.now(timezone.utc).isoformat(),
                })
                return None

            time.sleep(min(2 ** attempt, 15))

    return None


def unwrap_data(payload):
    if not isinstance(payload, dict):
        return None

    data = payload.get("data")

    if isinstance(data, (dict, list)):
        return data

    return payload


def get_post_list(payload):
    data = unwrap_data(payload)

    if isinstance(data, list):
        return data

    if isinstance(data, dict):
        for key in ("posts", "items", "results"):
            if isinstance(data.get(key), list):
                return data[key]

    return []


def get_total_pages(payload):
    if not isinstance(payload, dict):
        return None

    value = payload.get("totalPages")

    if value is None and isinstance(payload.get("meta"), dict):
        value = payload["meta"].get("totalPages")

    try:
        return int(value) if value is not None else None
    except (ValueError, TypeError):
        return None


# ---------- Date handling ----------

def parse_published_at(*sources):
    date_keys = (
        "publishTime",
        "publishedAt",
        "published_at",
        "createTime",
        "date",
    )

    for source in sources:
        if not isinstance(source, dict):
            continue

        for key in date_keys:
            value = source.get(key)
            if not value:
                continue

            try:
                ts = pd.Timestamp(value)

                if pd.isna(ts):
                    continue

                if ts.tzinfo is None:
                    ts = ts.tz_localize("Asia/Bangkok")
                else:
                    ts = ts.tz_convert("Asia/Bangkok")

                return ts.isoformat()

            except (ValueError, TypeError, OverflowError):
                continue

    return None


def in_date_range(published_at):
    # หากไม่ได้กำหนดช่วงเวลา ให้เก็บย้อนหลังทั้งหมด
    if START_DATE is None and END_DATE is None:
        return True

    # ถ้ากำหนดช่วงเวลา แต่ข่าวไม่มีวันที่ที่เชื่อถือได้
    # ให้ข้ามไว้ก่อนเพื่อลดความเสี่ยงนำข่าวผิดช่วงเข้าชุดข้อมูล
    if not published_at:
        return False

    ts = pd.Timestamp(published_at).date()

    if START_DATE and ts < pd.Timestamp(START_DATE).date():
        return False

    if END_DATE and ts > pd.Timestamp(END_DATE).date():
        return False

    return True


# ---------- Article detail ----------

def get_detail(site, post):
    rest_url = post.get("restUrl")
    post_id = post.get("id")

    if rest_url:
        detail_path = rest_url
    elif post_id is not None:
        detail_path = f"/v1/{site}/posts/{post_id}"
    else:
        return {}

    if detail_path in detail_cache:
        return detail_cache[detail_path]

    detail = api_get(detail_path)
    time.sleep(THAIPBS_DELAY)

    detail = unwrap_data(detail)

    if not isinstance(detail, dict):
        return {}

    # Cache เฉพาะข้อมูลรายละเอียด ไม่เก็บ response ที่ไม่ใช่บทความ
    keep = {
        key: detail.get(key)
        for key in (
            "id",
            "title",
            "abstract",
            "content",
            "publishTime",
            "createTime",
            "canonical",
            "tags",
            "categories",
        )
    }

    detail_cache[detail_path] = keep

    return keep


# ---------- Text cleaning ----------

import re
import html

ZW_RE = re.compile(r"[\u200b-\u200d\u2060\ufeff]")


def clean_text(value):
    if value is None:
        return ""

    text = str(value)

    # ลบ script/style และแปลง HTML เป็นข้อความ
    text = re.sub(
        r"(?is)<(script|style)[^>]*>.*?</\1>",
        " ",
        text,
    )
    text = re.sub(
        r"(?i)<br\s*/?>|</(p|div|li|h[1-6]|section|article)>",
        "\n",
        text,
    )
    text = re.sub(r"(?s)<[^>]+>", " ", text)

    text = html.unescape(text)
    text = ZW_RE.sub("", text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n\n", text)

    return text.strip()


# ---------- Collect posts from all configured sites ----------

def collect_site_posts(site):
    print(f"\nScanning site: {site}")

    page = 1
    total_pages = None
    seen_ids = set()
    collected = 0

    while page <= MAX_PAGES_PER_SITE:
        # news uses /v1/news/posts; other sites use /v1/{site}/posts
        endpoint = f"/v1/{site}/posts"

        payload = api_get(
            endpoint,
            params={"page": page, "limit": PAGE_SIZE},
        )

        if not payload:
            print(f"  Stopping at page {page}: empty/error response")
            break

        posts = get_post_list(payload)

        if total_pages is None:
            total_pages = get_total_pages(payload)

        if not posts:
            break

        for post in posts:
            if not isinstance(post, dict):
                continue

            post_id = post.get("id")
            canonical = post.get("canonical") or post.get("url")

            # Stable ID based on source and post ID.
            # If no ID exists, derive one from canonical URL or title.
            identity = (
                str(post_id)
                if post_id is not None
                else str(canonical or post.get("title") or "")
            )

            if not identity or identity in seen_ids:
                continue

            seen_ids.add(identity)

            detail = get_detail(site, post)

            title = clean_text(
                detail.get("title") or post.get("title") or ""
            )

            abstract = clean_text(
                detail.get("abstract") or post.get("abstract") or ""
            )

            raw_content = (
                detail.get("content")
                or detail.get("abstract")
                or post.get("content")
                or post.get("abstract")
                or ""
            )

            content = clean_text(raw_content)

            published_at = parse_published_at(post, detail)

            if not in_date_range(published_at):
                continue

            if not title and not content:
                continue

            tags = post.get("tags") or detail.get("tags") or []
            categories = (
                post.get("categories")
                or detail.get("categories")
                or []
            )

            article_id = f"thaipbs:{site}:{identity}"

            article = {
                "article_id": article_id,
                "source": {
                    "name": "Thai PBS",
                    "site": site,
                    "url": canonical or detail.get("canonical"),
                },
                "title": title,
                "abstract": abstract,
                "content": content,
                "published_at": published_at,
                "retrieved_at": datetime.now(timezone.utc).isoformat(),
                "tags": tags,
                "categories": categories,
                "content_status": (
                    "full_text"
                    if detail.get("content") or post.get("content")
                    else "abstract_only"
                ),
            }

            # Hash helps detect identical content across sources
            hash_input = (
                title.lower().strip()
                + "\n"
                + content.lower().strip()
            )
            article["content_hash"] = hashlib.sha256(
                hash_input.encode("utf-8")
            ).hexdigest()

            append_jsonl(RAW_FILE, article)
            collected += 1

            if collected % 50 == 0:
                save_json_cache(CACHE_FILE, detail_cache)
                print(f"  Saved {collected} new records from {site}")

        save_json_cache(CACHE_FILE, detail_cache)

        print(f"  Page {page}: {len(posts)} posts")

        if total_pages is not None and page >= total_pages:
            break

        page += 1
        time.sleep(THAIPBS_DELAY)

    print(f"Finished {site}: {collected} records saved")
    return collected


# ---------- Run collection with resume ----------

existing_raw = read_jsonl(RAW_FILE)
existing_article_ids = {
    item.get("article_id")
    for item in existing_raw
    if item.get("article_id")
}

print("Existing raw records:", len(existing_article_ids))

# To avoid appending duplicates when resuming, wrap append_jsonl.
_original_append_jsonl = append_jsonl

def append_jsonl(path, obj):
    if path == RAW_FILE:
        article_id = obj.get("article_id")
        if article_id in existing_article_ids:
            return

        existing_article_ids.add(article_id)

    _original_append_jsonl(path, obj)


new_records = 0

for site in SITES:
    new_records += collect_site_posts(site)

save_json_cache(CACHE_FILE, detail_cache)

print("\n===================================")
print("Collection finished")
print("New records:", new_records)
print("Total raw records:", len(read_jsonl(RAW_FILE)))
print("Raw JSONL:", RAW_FILE)
print("===================================")

In [2]:
# ========================================================
# CELL 3: THAILLM + LANGCHAIN CLASSIFICATION (PARALLEL)
# Fixed: Prompt Template, JSON Parsing, Resume & Retry
# Uses 7 API keys with mod7 partitioning to avoid race conditions
# ========================================================

import json
import re
import time
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock
from collections import defaultdict
from tqdm import tqdm

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# --------------------------------------------------------
# JSONL utility functions (needed for this cell)
# --------------------------------------------------------

def read_jsonl(path):
    """Read JSONL file and return list of dicts."""
    if not Path(path).exists():
        return []

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows


# --------------------------------------------------------
# 1. Configure Multiple ThaiLLM Instances (7 API keys)
# --------------------------------------------------------

num_keys = len(THAILLM_API_KEYS)
print(f"Initializing {num_keys} LLM instances for parallel processing")

llm_instances = []
for i, api_key in enumerate(THAILLM_API_KEYS):
    llm = ChatOpenAI(
        model=THAILLM_MODEL,
        base_url=THAILLM_BASE_URL,
        api_key=api_key,
        temperature=0,
        max_tokens=500,
        timeout=60,
        max_retries=2,
    )
    llm_instances.append(llm)
    print(f"  LLM instance {i+1}/{num_keys} ready")


# --------------------------------------------------------
# 2. Classification Prompt
# IMPORTANT:
# ห้ามใส่ตัวอย่าง JSON ที่มี { } แบบ literal ใน Prompt
# ใช้ชื่อ field อธิบายเป็นข้อความแทน
# ตัวแปรที่ต้องแทนค่าจริงยังใช้ {title} เป็นต้น
# --------------------------------------------------------

classification_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
คุณเป็นผู้ช่วยคัดกรองข่าวสำหรับระบบ Gold Market Research

หน้าที่คือพิจารณาว่าข่าวมีความเกี่ยวข้องกับตลาดทองคำจริงหรือไม่
ให้พิจารณาจากข้อมูลที่ได้รับเท่านั้น ห้ามแต่งข้อเท็จจริงเพิ่มเติม

เกณฑ์การจำแนก:

1. direct
ข่าวที่เกี่ยวข้องกับทองคำโดยตรง เช่น
- ราคาทองคำแท่ง ทองรูปพรรณ หรือ Gold Spot
- การซื้อขาย อุปสงค์ อุปทาน หรือกองทุนทองคำ
- การคาดการณ์หรือวิเคราะห์ตลาดทองคำ
- การซื้อหรือขายทองคำของธนาคารกลาง

2. indirect
ข่าวที่ไม่จำเป็นต้องกล่าวถึงทองคำโดยตรง แต่มีหลักฐานหรือเหตุผล
ที่เชื่อมโยงกับตลาดทองคำอย่างชัดเจน เช่น
- Fed, FOMC, อัตราดอกเบี้ย และนโยบายการเงิน
- เงินเฟ้อ CPI/PCE และข้อมูลเศรษฐกิจสำคัญ
- ค่าเงิน USD, DXY และ Bond Yield
- สงครามหรือความเสี่ยงภูมิรัฐศาสตร์ที่อาจกระทบ Safe-haven demand

ห้ามจัดข่าวเศรษฐกิจทั่วไปเป็น indirect เพียงเพราะอาจส่งผลต่อเศรษฐกิจ
ต้องมีเหตุผลเชื่อมโยงกับตลาดทองคำอย่างชัดเจน

3. not_relevant
ข่าวที่ไม่มีความเกี่ยวข้องกับตลาดทองคำอย่างมีนัยสำคัญ
เช่น ข่าวบันเทิง อาชญากรรมทั่วไป หรือข่าวท้องถิ่นที่ไม่เกี่ยวข้อง

4. uncertain
ข่าวสั้นเกินไป ข้อมูลขาดหาย หรือมีหลักฐานไม่เพียงพอที่จะตัดสิน

ข้อกำหนดผลลัพธ์:
- ตอบเป็น JSON object ที่ถูกต้องเท่านั้น
- ห้ามใช้ Markdown code fence
- ห้ามเพิ่มข้อความก่อนหรือหลัง JSON
- ต้องมี keys ต่อไปนี้:
  relevance, confidence, reason, evidence, market_topics
- relevance ต้องเป็นหนึ่งใน:
  direct, indirect, not_relevant, uncertain
- confidence เป็นจำนวนเต็มตั้งแต่ 0 ถึง 100
- reason เป็นเหตุผลสั้น ๆ ภาษาไทย
- evidence เป็น array ของข้อความหรือข้อเท็จจริงจากข่าว
- market_topics เป็น array ของหัวข้อที่เกี่ยวข้อง
- evidence ต้องอ้างจากข้อมูลข่าวที่ได้รับเท่านั้น
- หากไม่พบหลักฐาน ให้ใช้ array ว่าง
- หากไม่เกี่ยวข้องกับตลาดทองคำ ให้ market_topics เป็น array ว่าง
- อย่าสรุปความสัมพันธ์เชิงเหตุและผลเกินกว่าหลักฐานที่มี

ตัวอย่าง market_topics ที่อาจใช้ตามความเหมาะสม:
gold_price, gold_demand, central_bank_gold, Fed, interest_rate,
inflation, USD, DXY, bond_yield, geopolitics, oil, safe_haven

เลือกหัวข้อเฉพาะที่เกี่ยวข้องจริง ไม่จำเป็นต้องใส่ทุกหัวข้อ
        """.strip(),
    ),
    (
        "human",
        """
หัวข้อข่าว:
{title}

วันที่เผยแพร่:
{published_at}

บทคัดย่อ:
{abstract}

เนื้อหาข่าว:
{content}

โปรดจำแนกข่าวตามเกณฑ์ที่กำหนด
        """.strip(),
    ),
])

# Create classifier chains for each LLM instance
classifiers = [classification_prompt | llm | StrOutputParser() for llm in llm_instances]


# --------------------------------------------------------
# 3. Parse and Validate JSON
# --------------------------------------------------------

VALID_LABELS = {
    "direct",
    "indirect",
    "not_relevant",
    "uncertain",
}


def parse_llm_json(raw_text):
    """Parse JSON from ThaiLLM output and validate its fields."""

    text = str(raw_text or "").strip()

    # รองรับ Markdown code fence
    text = re.sub(
        r"^\s*```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\s*```\s*$", "", text)

    obj = None

    # วิธีที่ 1: ข้อความทั้งหมดเป็น JSON
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        pass

    # วิธีที่ 2: ค้นหา JSON object ภายในข้อความ
    if not isinstance(obj, dict):
        decoder = json.JSONDecoder()

        for match in re.finditer(r"\{", text):
            try:
                candidate, _ = decoder.raw_decode(
                    text[match.start():]
                )

                if isinstance(candidate, dict):
                    obj = candidate
                    break

            except json.JSONDecodeError:
                continue

    if not isinstance(obj, dict):
        raise ValueError(
            "ThaiLLM did not return a valid JSON object. "
            f"Output preview: {text[:300]}"
        )

    # Validate relevance
    relevance = str(
        obj.get("relevance", "uncertain")
    ).lower().strip()

    if relevance not in VALID_LABELS:
        relevance = "uncertain"

    # Validate confidence
    try:
        confidence = int(float(obj.get("confidence", 0)))
    except (ValueError, TypeError, OverflowError):
        confidence = 0

    confidence = max(0, min(100, confidence))

    # Validate evidence
    evidence = obj.get("evidence", [])

    if not isinstance(evidence, list):
        evidence = []

    # Validate market topics
    topics = obj.get("market_topics", [])

    if not isinstance(topics, list):
        topics = []

    # ป้องกันข้อมูลประเภท object/list ที่ไม่เหมาะสมใน text fields
    reason = obj.get("reason", "")

    if not isinstance(reason, str):
        reason = str(reason)

    return {
        "relevance": relevance,
        "confidence": confidence,
        "reason": reason,
        "evidence": [str(x) for x in evidence],
        "market_topics": [str(x) for x in topics],
    }


# --------------------------------------------------------
# 4. Classify One Article (Thread-safe)
# --------------------------------------------------------

def get_llm_index(article_id):
    """Hash article_id to determine which LLM instance to use (mod7)."""
    hash_value = int(hashlib.md5(article_id.encode()).hexdigest(), 16)
    return hash_value % num_keys


def classify_article(article, llm_index):
    """Classify a single article using a specific LLM instance."""
    title = str(article.get("title") or "")
    abstract = str(article.get("abstract") or "")
    content = str(article.get("content") or "")

    # ควบคุมความยาว input
    abstract_for_llm = abstract[:2000]
    content_for_llm = content[:7000]

    if not title.strip() and not content_for_llm.strip():
        return {
            "relevance": "uncertain",
            "confidence": 0,
            "reason": "ไม่มีหัวข้อและเนื้อหาข่าวเพียงพอ",
            "evidence": [],
            "market_topics": [],
        }

    last_error = None
    classifier = classifiers[llm_index]

    # Retry สูงสุด 3 ครั้งต่อข่าว
    for attempt in range(3):
        try:
            response = classifier.invoke({
                "title": title,
                "published_at": (
                    article.get("published_at") or "ไม่ทราบ"
                ),
                "abstract": abstract_for_llm,
                "content": content_for_llm,
            })

            return parse_llm_json(response)

        except Exception as exc:
            last_error = str(exc)

            # ไม่ต้องรอนานหากเป็น Template หรือข้อมูลผิดพลาดเดิม
            wait_seconds = min(2 ** (attempt + 1), 15)

            print(
                f"[LLM-{llm_index+1}] Classification retry {attempt + 1}/3 | "
                f"{article.get('article_id')} | "
                f"{last_error[:250]}"
            )

            if attempt < 2:
                time.sleep(wait_seconds)

    # เก็บ error ไว้เพื่อให้รอบถัดไป retry ได้
    return {
        "relevance": "uncertain",
        "confidence": 0,
        "reason": "Classification failed; requires retry or review",
        "evidence": [],
        "market_topics": [],
        "error": last_error or "Unknown classification error",
    }


# --------------------------------------------------------
# 5. JSONL File Helpers (Thread-safe)
# --------------------------------------------------------

def write_jsonl_atomic(filepath, records):
    """
    เขียนไฟล์ใหม่ผ่าน temporary file ก่อนแทนที่ไฟล์เดิม
    ช่วยลดความเสี่ยงที่ไฟล์จะเสียหากการเขียนถูกขัดจังหวะ
    """

    path = Path(filepath)
    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = path.with_name(path.name + ".tmp")

    with temp_path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )

    temp_path.replace(path)


# Thread-safe append with lock
file_lock = Lock()

def append_jsonl_record(filepath, record):
    """Thread-safe append exactly one record."""
    with file_lock:
        path = Path(filepath)
        path.parent.mkdir(parents=True, exist_ok=True)

        with path.open("a", encoding="utf-8") as file:
            file.write(
                json.dumps(record, ensure_ascii=False) + "\n"
            )


def is_classification_complete(record):
    """
    uncertain ถือเป็นผลที่จำแนกเสร็จได้
    หากไม่มี error และมี label ที่ถูกต้อง
    """

    if not isinstance(record, dict):
        return False

    classification = record.get("classification")

    if not isinstance(classification, dict):
        return False

    if classification.get("error"):
        return False

    return (
        classification.get("relevance") in VALID_LABELS
    )


# --------------------------------------------------------
# 6. Resume from Previous Classifications
# --------------------------------------------------------

raw_articles = read_jsonl(RAW_FILE)

try:
    existing_classified = read_jsonl(CLASSIFIED_FILE)
except FileNotFoundError:
    existing_classified = []


# เก็บเฉพาะผลที่จำแนกสำเร็จแล้ว
# รายการที่มี error จะไม่ถูกนับว่าเสร็จ
existing_by_id = {}
existing_order = []
records_without_id = []

for item in existing_classified:
    article_id = item.get("article_id")

    if not article_id:
        records_without_id.append(item)
        continue

    if not is_classification_complete(item):
        continue

    if article_id not in existing_by_id:
        existing_order.append(article_id)

    # กรณีมี ID ซ้ำ ให้เก็บรายการที่พบล่าสุด
    existing_by_id[article_id] = item


# สร้างไฟล์ใหม่ครั้งเดียวก่อนเริ่มงาน
# วิธีนี้นำ error records เก่าออก เพื่อให้สามารถ retry ได้จริง
cleaned_existing = (
    records_without_id
    + [
        existing_by_id[article_id]
        for article_id in existing_order
    ]
)

if cleaned_existing != existing_classified:
    write_jsonl_atomic(
        CLASSIFIED_FILE,
        cleaned_existing,
    )

classified_ids = set(existing_by_id.keys())

raw_ids = {
    item.get("article_id")
    for item in raw_articles
    if item.get("article_id")
}

remaining_ids = raw_ids - classified_ids

print("Raw articles:", len(raw_articles))
print("Unique raw article IDs:", len(raw_ids))
print("Already classified successfully:", len(classified_ids))
print("Remaining / will retry:", len(remaining_ids))
print("Classified JSONL:", CLASSIFIED_FILE)


# --------------------------------------------------------
# 7. Run Parallel Classification
# --------------------------------------------------------

processed = 0
succeeded = 0
failed = 0

# ป้องกันการประมวลผลซ้ำกรณี RAW_FILE มี article_id ซ้ำ
attempted_ids = set(classified_ids)

# Filter articles that need classification
articles_to_classify = [
    article for article in raw_articles
    if article.get("article_id") and article.get("article_id") not in classified_ids
]

print(f"\nStarting parallel classification with {num_keys} workers")
print(f"Articles to classify: {len(articles_to_classify)}")


def process_article(article):
    """Process a single article and return result with metadata."""
    article_id = article.get("article_id")
    
    # Skip if already attempted (double-check for thread safety)
    if article_id in attempted_ids:
        return None
    
    # Determine which LLM instance to use based on article_id hash
    llm_index = get_llm_index(article_id)
    
    # Classify
    result = classify_article(article, llm_index)
    
    # Build record
    record = {
        **article,
        "classification": result,
        "classification_metadata": {
            "provider": "ThaiLLM",
            "model": THAILLM_MODEL,
            "framework": "LangChain",
            "classified_at": datetime.now(timezone.utc).isoformat(),
            "prompt_version": "gold-relevance-v2",
            "llm_instance": llm_index + 1,
        },
    }
    
    return record


# Use ThreadPoolExecutor for parallel processing
max_workers = num_keys  # One worker per API key

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    # Submit all tasks
    future_to_article = {
        executor.submit(process_article, article): article
        for article in articles_to_classify
    }
    
    # Process results as they complete with tqdm progress bar
    with tqdm(total=len(articles_to_classify), desc="Classifying", unit="article") as pbar:
        for future in as_completed(future_to_article):
            article = future_to_article[future]
            article_id = article.get("article_id")
            
            try:
                record = future.result()
                
                if record is None:
                    # Already processed by another thread
                    pbar.update(1)
                    continue
                
                # Write immediately with thread-safe append
                append_jsonl_record(CLASSIFIED_FILE, record)
                
                attempted_ids.add(article_id)
                processed += 1
                
                if is_classification_complete(record):
                    classified_ids.add(article_id)
                    succeeded += 1
                else:
                    failed += 1
                    
                    # Save error separately
                    result = record.get("classification", {})
                    append_jsonl_record(ERROR_FILE, {
                        "stage": "classification",
                        "article_id": article_id,
                        "error": result.get("error", "Classification failed"),
                        "timestamp": datetime.now(timezone.utc).isoformat(),
                    })
                
                # Update progress bar with stats
                result = record.get("classification", {})
                pbar.set_postfix({
                    'success': succeeded,
                    'failed': failed,
                    'latest': result.get('relevance')
                })
                pbar.update(1)
                
                # Rate limiting per worker
                time.sleep(THAILLM_DELAY)
                
            except Exception as exc:
                print(f"Error processing {article_id}: {exc}")
                failed += 1
                pbar.update(1)


# --------------------------------------------------------
# 8. Final Summary
# --------------------------------------------------------

print("\n========== Classification Finished ==========")
print("Processed this run:", processed)
print("Successful this run:", succeeded)
print("Failed this run:", failed)
print("Successfully classified IDs:", len(classified_ids))
print("Classified JSONL:", CLASSIFIED_FILE)
print("Error JSONL:", ERROR_FILE)

if failed > 0:
    print(
        "\nNote: Failed records will be eligible for retry "
        "when you run this cell again."
    )

Initializing 7 LLM instances for parallel processing
  LLM instance 1/7 ready
  LLM instance 2/7 ready
  LLM instance 3/7 ready
  LLM instance 4/7 ready
  LLM instance 5/7 ready
  LLM instance 6/7 ready
  LLM instance 7/7 ready
Raw articles: 7874
Unique raw article IDs: 7874
Already classified successfully: 1907
Remaining / will retry: 5967
Classified JSONL: D:\Project_Github\gold assisstant\dataset\news\thaipbs_classified.jsonl

Starting parallel classification with 7 workers
Articles to classify: 5967


Classifying:   0%|          | 0/5967 [00:00<?, ?article/s]

[LLM-4] Classification retry 1/3 | thaipbs:news:571542 | ThaiLLM did not return a valid JSON object. Output preview: <think>
ข่าวที่ให้มาเกี่ยวกับ "มหกรรมหนังสือระดับชาติ ครั้งที่ 31" ซึ่งเป็นงานที่รวบรวมหนังสือจากสำนักพิมพ์ต่างๆ และมีกิจกรรมส่งเสริมการอ่าน แต่ไม่ได้กล่าวถึงทองคำโดยตรง หรือมีเหตุผลเชื่อมโย[LLM-3] Classification retry 1/3 | thaipbs:news:571541 | ThaiLLM did not return a valid JSON object. Output preview: <think>
ข่าวที่ให้มาเกี่ยวข้องกับสถานการณ์ทางการเมืองและภูมิรัฐศาสตร์ในตะวันออกกลาง โดยเฉพาะความตึงเครียดระหว่างสหรัฐฯ กับซาอุดีอาระเบีย และกลุ่มฮูตีในเยเมนที่ได้รับการสนับสนุนจากอิหร่าน แม้
[LLM-7] Classification retry 1/3 | thaipbs:news:571544 | ThaiLLM did not return a valid JSON object. Output preview: <think>
ข่าวเกี่ยวกับการโจมตีสนามบินคิงฟาฮัดในซาอุดีอาระเบีย ทำให้เกิดความเสียหายและผู้เสียชีวิต สถานทูตไทยเตือนคนไทยหลีกเลี่ยงพื้นที่ดังกล่าว ข่าวไม่ได้กล่าวถึงทองคำโดยตรง แต่มีความเชื่อมโย

[LLM-3] Classification retry 1/3 | thaipbs:news:571536 | ThaiLLM did not ret

Classifying:   0%|          | 1/5967 [00:11<18:14:03, 11.00s/article, success=1, failed=0, latest=not_relevant]

[LLM-4] Classification retry 2/3 | thaipbs:news:571540 | ThaiLLM did not return a valid JSON object. Output preview: <think>
ข่าวที่ให้มาเกี่ยวกับการแต่งตั้ง "อุบลรัตน์ เถาว์น้อย" เป็นผู้ช่วยโฆษกรัฐบาล ไม่ได้กล่าวถึงตลาดทองคำโดยตรง แต่ต้องพิจารณาว่ามีความเกี่ยวข้องทางอ้อมหรือไม่

ข่าวไม่ได้กล่าวถึงนโยบายเศ[LLM-7] Classification retry 2/3 | thaipbs:news:571544 | ThaiLLM did not return a valid JSON object. Output preview: <think>
ข่าวเกี่ยวกับการโจมตีสนามบินคิงฟาฮัดในซาอุดีอาระเบีย ทำให้เกิดความเสียหายและผู้เสียชีวิต สถานทูตไทยเตือนคนไทยหลีกเลี่ยงพื้นที่ดังกล่าว ข่าวไม่ได้กล่าวถึงทองคำโดยตรง แต่มีความเชื่อมโย
[LLM-5] Classification retry 2/3 | thaipbs:news:571539 | ThaiLLM did not return a valid JSON object. Output preview: <think>
จากข่าวที่ให้มา ประเด็นหลักคือการจับกุมนักธุรกิจชาวจีนที่หลบหนีหมายจับในคดีทุจริตภาษี โดยมีการระบุว่าเธอซ่อนตัวในประเทศไทย แม้ว่าข่าวจะไม่ได้กล่าวถึงทองคำโดยตรง แต่มีประเด็นที่เกี่ยว

[LLM-3] Classification retry 2/3 | thaipbs:news:571541 | ThaiLLM did not ret

Classifying:   0%|          | 3/5967 [00:18<8:28:20,  5.11s/article, success=2, failed=1, latest=uncertain]    

... (10,895 outputs left out)

Classifying: 100%|██████████| 5967/5967 [6:17:58<00:00,  2.01s/article, success=793, failed=5174, latest=uncertain]

[LLM-5] Classification retry 3/3 | thaipbs:theactive:57282 | ThaiLLM did not return a valid JSON object. Output preview: <think>
ข่าวที่ให้มาเกี่ยวกับกากแร่แคดเมียมที่สูญหายและพบซุกซ่อนในจังหวัดชลบุรี ประเด็นหลักคือการขนส่งและการจัดการของเสียอันตรายที่ไม่ถูกต้องตามกฎหมาย แม้ว่าแคดเมียมจะเป็นโลหะหนักที่เป็นอันต


Classifying: 100%|██████████| 5967/5967 [6:17:58<00:00,  3.80s/article, success=793, failed=5174, latest=uncertain]


========== Classification Finished ==========
Processed this run: 5967
Successful this run: 793
Failed this run: 5174
Successfully classified IDs: 2700
Classified JSONL: D:\Project_Github\gold assisstant\dataset\news\thaipbs_classified.jsonl
Error JSONL: D:\Project_Github\gold assisstant\dataset\news\collection_errors.jsonl

Note: Failed records will be eligible for retry when you run this cell again.


In [3]:
# ========================================================
# CELL 4: BUILD FINAL DATASETS + QUALITY CHECK
# ========================================================
from collections import Counter
import json
from pathlib import Path


def read_jsonl(path):
    """Read JSONL file and return list of dicts."""
    if not Path(path).exists():
        return []

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows


all_classified = read_jsonl(CLASSIFIED_FILE)

gold_records = []
uncertain_records = []

for item in all_classified:
    result = item.get("classification") or {}
    relevance = result.get("relevance", "uncertain")

    if relevance in ("direct", "indirect"):
        gold_records.append(item)
    elif relevance == "uncertain":
        uncertain_records.append(item)


def write_jsonl(path, records):
    # เขียนใหม่ทุกครั้ง เพื่อไม่ให้เกิด record ซ้ำ
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


UNCERTAIN_FILE = OUT_DIR / "gold_uncertain_news.jsonl"

write_jsonl(GOLD_FILE, gold_records)
write_jsonl(UNCERTAIN_FILE, uncertain_records)

label_counts = Counter(
    (item.get("classification") or {}).get("relevance", "uncertain")
    for item in all_classified
)

status_counts = Counter(
    item.get("content_status", "unknown")
    for item in all_classified
)

print("=" * 55)
print("GOLD ASSISTANT — NEWS DATASET REPORT")
print("=" * 55)
print(f"Raw records        : {len(read_jsonl(RAW_FILE))}")
print(f"Classified records : {len(all_classified)}")
print(f"Direct gold news   : {label_counts['direct']}")
print(f"Indirect gold news : {label_counts['indirect']}")
print(f"Not relevant       : {label_counts['not_relevant']}")
print(f"Uncertain          : {label_counts['uncertain']}")
print("-" * 55)
print("Content completeness:")
for key, count in status_counts.items():
    print(f"  {key}: {count}")
print("-" * 55)
print("Gold-related JSONL :", GOLD_FILE)
print("Uncertain JSONL    :", UNCERTAIN_FILE)
print("All classified     :", CLASSIFIED_FILE)
print("Errors             :", ERROR_FILE)
print("=" * 55)

# แสดงตัวอย่างข่าวทองที่ผ่านการคัดกรอง
for item in gold_records[:10]:
    result = item.get("classification") or {}
    print(f"\n[{result.get('relevance')}] {item.get('title')}")
    print(f"Published: {item.get('published_at')}")
    print(f"Confidence: {result.get('confidence')}")
    print(f"Reason: {result.get('reason')}")
    print(f"URL: {(item.get('source') or {}).get('url')}")

GOLD ASSISTANT — NEWS DATASET REPORT
Raw records        : 7874
Classified records : 7874
Direct gold news   : 9
Indirect gold news : 919
Not relevant       : 1723
Uncertain          : 5223
-------------------------------------------------------
Content completeness:
  full_text: 7874
-------------------------------------------------------
Gold-related JSONL : D:\Project_Github\gold assisstant\dataset\news\gold_related_news.jsonl
Uncertain JSONL    : D:\Project_Github\gold assisstant\dataset\news\gold_uncertain_news.jsonl
All classified     : D:\Project_Github\gold assisstant\dataset\news\thaipbs_classified.jsonl
Errors             : D:\Project_Github\gold assisstant\dataset\news\collection_errors.jsonl

[indirect] "นรเศรษฐ์" รับหนังสือค้านร่างกฎหมาย 4 ฉบับ หวั่นกระทบความเป็นอิสระขององค์กร
Published: 2026-10-21T11:29:37+07:00
Confidence: 80
Reason: ข่าวเกี่ยวข้องกับความเสี่ยงทางการเมืองและผลกระทบต่อองค์กรสื่อสาธารณะ ซึ่งอาจส่งผลต่อความต้องการทองคำในฐานะสินทรัพย์ปลอดภัย
URL: https://www.

# Clean data

In [6]:
import json
from collections import Counter

FILE_PATH = (
    r"D:\Project_Github\gold assisstant\dataset\news"
    r"\thaipbs_classified.jsonl"
)

LABELS = [
    "direct",
    "indirect",
    "not_relevant",
    "uncertain",
]

records = []

with open(FILE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

print("Total records:", len(records))
print("\nClassification distribution:")

counts = Counter(
    (r.get("classification") or {}).get("relevance", "missing")
    for r in records
)

for label, count in counts.most_common():
    print(f"{label:15s} {count:5,}")

# แสดงตัวอย่าง 5 ข่าวต่อประเภท
for label in LABELS:
    samples = [
        r for r in records
        if (r.get("classification") or {}).get("relevance") == label
    ]

    print("\n" + "=" * 80)
    print(f"LABEL: {label} | Total: {len(samples):,}")
    print("=" * 80)

    for i, r in enumerate(samples[:5], 1):
        c = r.get("classification") or {}

        print(f"\n--- Example {i} ---")
        print("Title:", r.get("title", ""))
        print("Published:", r.get("published_at", ""))
        print("Confidence:", c.get("confidence"))
        print("Reason:", c.get("reason"))
        print("Evidence:", json.dumps(
            c.get("evidence", []), ensure_ascii=False
        ))
        print("Topics:", json.dumps(
            c.get("market_topics", []), ensure_ascii=False
        ))
        print("URL:", (r.get("source") or {}).get("url", ""))

Total records: 7874

Classification distribution:
uncertain       5,223
not_relevant    1,723
indirect          919
direct              9

LABEL: direct | Total: 9

--- Example 1 ---
Title: สรุปราคาทองคำ 20 ต.ค.2569 ปิดตลาด ผันผวน 21 ครั้ง ลบ 650 บาท
Published: 2026-10-20T17:52:29+07:00
Confidence: 90
Reason: ข่าวรายงานราคาทองคำแท่งและทองรูปพรรณที่เปลี่ยนแปลงตลอดวัน รวมถึงการผันผวนของราคา
Evidence: ["ทองแท่ง รับซื้อ บาทละ 65,400 บาท ขายออก บาทละ 65,600 บาท", "ทองรูปพรรณ รับซื้อ บาทละ 64,096.48 บาท ขายออก บาทละ 66,400 บาท", "ผันผวน 21 ครั้ง ลบ 650 บาท"]
Topics: ["ราคาทองคำ", "ทองแท่ง", "ทองรูปพรรณ", "การผันผวนของราคา"]
URL: https://www.thaipbs.or.th/news/content/571381

--- Example 2 ---
Title: สรุปราคาทองคำ 23 ต.ค.2569 ปิดตลาด ผันผวน 24 ครั้ง ลบ 1,000 บาท
Published: 2026-10-23T17:50:49+07:00
Confidence: 95
Reason: ข่าวรายงานราคาทองคำแท่งและทองรูปพรรณในประเทศ รวมถึงการวิเคราะห์แนวโน้มราคาทองคำโลก
Evidence: ["ราคาทองคำปรับตัวลง -116.7 ดอลลาร์", "ทองคำแท่งขายออก บาทละ 64,350 บาท", "ทองรูป

In [5]:
# ========================================================
# CELL 7: AUDITOR CORE — STRICT PARSER + SCHEMA VALIDATION + TESTS
# แก้เฉพาะการรับ output / retry (ไม่เปลี่ยนเกณฑ์การจำแนกข่าว)
# - Balanced-brace parser: รองรับ JSON ตรง, Markdown fence,
#   <think> block, ข้อความนำหน้า, JSON หลาย object, ข้อความที่ถูกตัด
# - Schema validation + typed technical errors (ไม่ใช่ semantic uncertainty)
# - Unit tests ด้วย mock output (ไม่เรียก API)
# ========================================================

import json
import re
from pathlib import Path


# --------------------------------------------------------
# 1. Paths (ไฟล์ต้นฉบับห้ามเขียนทับ)
# --------------------------------------------------------
AUDIT_V1_FILE = OUT_DIR / "gold_relevance_audit.jsonl"      # ต้นฉบับ (อ่านเท่านั้น)
AUDIT_V2_FILE = OUT_DIR / "gold_relevance_audit_v2.jsonl"   # ผลลัพธ์เวอร์ชันใหม่ (เขียน)
AUDIT_RETRY_LOG = OUT_DIR / "audit_retry_errors.jsonl"      # log ของ technical failure

AUDIT_LABELS = ("direct", "indirect", "not_relevant", "uncertain")

# ประเภทความล้มเหลวทางเทคนิค — ต้องไม่ถูกแปลงเป็น label ทาง semantic
TECHNICAL_ERROR_TYPES = (
    "api_error",     # HTTP / transport error อื่น ๆ
    "timeout",       # หมดเวลา
    "rate_limit",    # 429 / too many requests
    "truncated",     # output ถูกตัดกลางข้อความ (> max_tokens)
    "parse_error",   # ไม่พบ JSON object ที่ใช้งานได้
    "schema_error",  # พบ JSON แต่ field/ชนิดข้อมูลไม่ตรง schema
)


# --------------------------------------------------------
# 2. JSONL file helpers (atomic write)
# --------------------------------------------------------

def read_jsonl_file(path):
    """Read JSONL file and return list of dicts."""
    if not Path(path).exists():
        return []

    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    return rows


def write_jsonl_atomic(path, records):
    """เขียนทับทั้งไฟล์ผ่าน temporary file (กันไฟล์เสียกลางคัน)"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = path.with_name(path.name + ".tmp")

    with temp_path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    temp_path.replace(path)


def append_jsonl(path, record):
    """ต่อท้าย 1 record (ทนต่อการหยุดกลางคัน)"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def dedupe_by_article_id(records):
    """คืน dict keyed by article_id แบบ last-wins (กัน record ซ้ำ)"""
    out = {}
    for record in records:
        article_id = record.get("article_id")
        if article_id:
            out[article_id] = record
    return out


# --------------------------------------------------------
# 3. Technical error type
# --------------------------------------------------------

class AuditFormatError(Exception):
    """ความล้มเหลวทางเทคนิคในการอ่าน output (ไม่ใช่ semantic uncertainty)."""

    def __init__(self, kind, message, preview=""):
        super().__init__(message)
        if kind not in TECHNICAL_ERROR_TYPES:
            raise ValueError(f"unknown error kind: {kind}")
        self.kind = kind
        self.preview = str(preview)[:300]


def classify_api_error(exc):
    """แยก api error / timeout / rate limit จาก exception"""
    message = str(exc).lower()

    if "timeout" in message or "timed out" in message:
        return "timeout"
    if "429" in message or "rate limit" in message or "too many requests" in message:
        return "rate_limit"
    return "api_error"


def scrub_secrets(text):
    """ลบ API key / secret ออกจากข้อความก่อน log"""
    out = str(text)
    for key in THAILLM_API_KEYS:
        if key and len(key) >= 8:
            out = out.replace(key, "***")
    out = re.sub(r"sk-[A-Za-z0-9_\-]{8,}", "***", out)
    out = re.sub(r"Bearer\s+[A-Za-z0-9_\-\.]{8,}", "Bearer ***", out)
    return out


# --------------------------------------------------------
# 4. Reasoning block + balanced-brace scanner
# --------------------------------------------------------
_THINK_BLOCK_RE = re.compile(r"<think\b[^>]*>.*?</think\s*>", re.IGNORECASE | re.DOTALL)
_THINK_OPEN_RE = re.compile(r"<think\b[^>]*>", re.IGNORECASE)
_THINK_CLOSE_RE = re.compile(r"</think\s*>", re.IGNORECASE)


def strip_reasoning(text):
    """ตัด reasoning block ที่ปิดครบแล้วออก

    คืน (ข้อความที่ตัดแล้ว, มี <think> ที่ไม่ปิดหรือไม่)
    """
    cleaned = _THINK_BLOCK_RE.sub(" ", text)
    unclosed_think = bool(_THINK_OPEN_RE.search(cleaned)) and not _THINK_CLOSE_RE.search(cleaned)

    if unclosed_think:
        cleaned = _THINK_OPEN_RE.sub(" ", cleaned)

    return cleaned, unclosed_think


def scan_json_objects(text):
    """สแกนหา JSON object ที่วงเล็บปีกกาสมดุล

    ใช้ตัวนับวงเล็บจริง (ไม่ใช่ regex) จึงรองรับ
    - เครื่องหมาย { } ที่อยู่ภายใน string
    - escaped quotation mark (\\")
    - ข้อความหรือ code fence ที่อยู่รอบ ๆ

    คืน (list ของ raw JSON string, truncated flag)
    truncated = True เมื่อพบ { ที่ไม่ถูกปิด (output ถูกตัดกลางข้อความ)
    """
    objects = []
    truncated = False

    length = len(text)
    i = 0

    while i < length:
        if text[i] != "{":
            i += 1
            continue

        depth = 0
        in_string = False
        escaped = False
        end = None

        j = i
        while j < length:
            ch = text[j]

            if in_string:
                if escaped:
                    escaped = False
                elif ch == "\\":
                    escaped = True
                elif ch == '"':
                    in_string = False
            else:
                if ch == '"':
                    in_string = True
                elif ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        end = j
                        break

            j += 1

        if end is None:
            # วงเล็บสุดท้ายไม่สมดุล = ถูกตัดกลางข้อความ
            truncated = True
            break

        objects.append(text[i:end + 1])
        i = end + 1

    return objects, truncated


def select_final_answer(raw_objects):
    """เลือก final answer อย่างชัดเจน

    กติกา: เลือก JSON object ตัวสุดท้ายที่มี key "relevance"
    (ตัวอย่างหรือ reasoning ที่โมเดลแสดงไว้จะอยู่ก่อนคำตอบจริง)
    ห้ามเลือก object ที่ไม่มี field relevance
    """
    chosen = None

    for raw in raw_objects:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            continue

        if isinstance(obj, dict) and "relevance" in obj:
            chosen = obj

    return chosen


# --------------------------------------------------------
# 5. Schema validation
# --------------------------------------------------------

def validate_audit_schema(obj):
    """ตรวจ schema และค่าของทุก field

    คืน (result dict, list ของ error)
    result จะเป็น None ใน field ที่ไม่ผ่านการตรวจ
    """
    errors = []

    if not isinstance(obj, dict):
        return None, ["result is not a JSON object"]

    relevance = obj.get("relevance")
    if isinstance(relevance, str) and relevance.strip().lower() in AUDIT_LABELS:
        relevance = relevance.strip().lower()
    else:
        errors.append(f"invalid relevance: {relevance!r}")
        relevance = None

    confidence = obj.get("confidence")
    try:
        confidence = int(float(confidence))
    except (TypeError, ValueError, OverflowError):
        errors.append(f"invalid confidence: {obj.get('confidence')!r}")
        confidence = None
    else:
        if not 0 <= confidence <= 100:
            errors.append(f"confidence out of range: {confidence}")
            confidence = None

    reason = obj.get("reason")
    if not isinstance(reason, str):
        errors.append("reason is not a string")
        reason = None

    evidence = obj.get("evidence")
    if not isinstance(evidence, list) or any(not isinstance(x, str) for x in evidence):
        errors.append("evidence is not a list of strings")
        evidence = None

    market_link = obj.get("market_link")
    if not isinstance(market_link, str):
        errors.append("market_link is not a string")
        market_link = None

    review_required = obj.get("review_required")
    if not isinstance(review_required, bool):
        errors.append("review_required is not a boolean")
        review_required = None

    result = {
        "relevance": relevance,
        "confidence": confidence,
        "reason": reason,
        "evidence": evidence,
        "market_link": market_link,
        "review_required": review_required,
    }

    return result, errors


# --------------------------------------------------------
# 6. Parser
# --------------------------------------------------------

def parse_audit_json(raw_text):
    """แปลง output ของโมเดลเป็น audit result ที่ผ่าน schema

    ลำดับการทำงาน:
    1. ตัด reasoning block ที่ปิดครบแล้ว (<think>...</think>) ออก
    2. สแกนหา JSON object ที่วงเล็บสมดุลทั้งหมด
    3. ถ้าพบวงเล็บที่ถูกตัดกลางข้อความ -> truncated (ต้อง retry)
    4. เลือก final answer = object ตัวสุดท้ายที่มี key relevance
    5. ตรวจ schema และค่าทุก field

    โยน AuditFormatError (technical failure) แทนการเดา label
    """
    text = str(raw_text or "")

    if not text.strip():
        raise AuditFormatError("parse_error", "empty output", "")

    cleaned, unclosed_think = strip_reasoning(text)
    objects, truncated = scan_json_objects(cleaned)

    if truncated:
        raise AuditFormatError(
            "truncated",
            "output ends with an unbalanced '{' (truncated by max_tokens)",
            text,
        )

    if not objects:
        kind = "truncated" if unclosed_think else "parse_error"
        raise AuditFormatError(kind, "no JSON object found in output", text)

    obj = select_final_answer(objects)
    if obj is None:
        raise AuditFormatError(
            "schema_error",
            "no JSON object contains field 'relevance'",
            text,
        )

    result, errors = validate_audit_schema(obj)
    if errors:
        raise AuditFormatError("schema_error", "; ".join(errors), text)

    result["status"] = "ok"
    return result


# --------------------------------------------------------
# 7. Audit state (seeded จาก v1 โดยไม่ทับผลที่สำเร็จแล้ว)
# --------------------------------------------------------

def is_audit_ok(record):
    """ผล audit ผ่าน schema/สถานะสำเร็จหรือไม่"""
    audit = record.get("audit")

    if not isinstance(audit, dict):
        return False
    if audit.get("status") != "ok":
        return False
    if audit.get("error"):
        return False

    return (
        audit.get("relevance") in AUDIT_LABELS
        and isinstance(audit.get("confidence"), int)
        and isinstance(audit.get("reason"), str)
        and isinstance(audit.get("evidence"), list)
        and isinstance(audit.get("market_link"), str)
        and isinstance(audit.get("review_required"), bool)
    )


def normalize_v1_audit(audit):
    """แปลงผล audit จาก v1 ให้เป็นรูปแบบ v2 (เพิ่ม status)

    คืน None ถ้า record นั้นเป็น technical failure หรือไม่ผ่าน schema
    """
    if not isinstance(audit, dict):
        return None
    if audit.get("error"):
        return None

    passed = (
        audit.get("relevance") in AUDIT_LABELS
        and isinstance(audit.get("confidence"), int)
        and isinstance(audit.get("reason"), str)
        and isinstance(audit.get("evidence"), list)
        and isinstance(audit.get("market_link"), str)
        and isinstance(audit.get("review_required"), bool)
    )
    if not passed:
        return None

    out = dict(audit)
    out["status"] = "ok"
    return out


def load_v2_state():
    """โหลดผลที่สำเร็จแล้วจาก v2 (authoritative) + seed จาก v1

    ห้ามลบหรือเขียนทับผล audit ที่สำเร็จแล้ว
    """
    state = dedupe_by_article_id(read_jsonl_file(AUDIT_V2_FILE))
    seeded = 0

    for record in read_jsonl_file(AUDIT_V1_FILE):
        article_id = record.get("article_id")
        if not article_id or article_id in state:
            continue

        audit = normalize_v1_audit(record.get("audit"))
        if audit is None:
            continue

        merged = dict(record)
        merged["audit"] = audit
        state[article_id] = merged
        seeded += 1

    return state, seeded


def find_pending_ids(all_records, state):
    """หา article_id ที่ยังไม่มีผล audit ที่ผ่าน schema (ต้อง retry)

    ใช้ article_id เป็น key จับคู่ผล ห้ามอาศัยลำดับบรรทัด
    """
    pending = []
    seen = set()

    for record in all_records:
        article_id = record.get("article_id")
        if not article_id or article_id in seen:
            continue
        seen.add(article_id)

        if article_id in state and is_audit_ok(state[article_id]):
            continue

        pending.append(article_id)

    return pending


# --------------------------------------------------------
# 8. Auditor prompt
#    ใช้ระบบ "strict prompt + parser + schema validation"
#    เพราะ endpoint ThaiLLM ไม่ได้เปิด native JSON mode
#    (ตรวจแล้ว: ใช้ ChatOpenAI + StrOutputParser ไม่มี response_format)
# --------------------------------------------------------
AUDITOR_SYSTEM_PROMPT = """
# OUTPUT DISCIPLINE (HIGHEST PRIORITY — DO NOT VIOLATE)
- Respond with exactly ONE JSON object and nothing else.
- Do NOT output reasoning, thinking, or any think block.
- Do NOT use Markdown code fences.
- Do NOT output any text before or after the JSON object.
- The response must be parseable JSON on the first attempt.

# ROLE
You are a strict financial-news relevance auditor for a Thai gold-market research and RAG system.

Your task is to determine whether a Thai PBS news article is DIRECTLY or INDIRECTLY relevant to the gold market, using only evidence supported by the article.

You are not a general-purpose news analyst. Do not invent causal relationships, market reactions, investor behavior, or gold-price implications that are not supported by the article.

# OBJECTIVE
Audit articles previously classified as either `direct` or `indirect`.

Choose exactly one result:
- `direct`
- `indirect`
- `not_relevant`
- `uncertain`

Do not force an article into `direct` or `indirect` if the evidence is insufficient.

# DEFINITIONS

## 1. DIRECT
Use `direct` when the article's main subject explicitly concerns gold or the gold market.

Qualifying examples:
- Thai gold prices, gold-price announcements, gold price changes, and daily gold-market summaries.
- Gold trading, gold demand or supply, gold investment, gold holdings, or gold-market transactions.
- Gold Spot or other gold-market indicators explicitly discussed in relation to gold prices.
- Gold-market analysis in which gold prices, gold demand, or gold trading are central subjects.

A passing mention of gold is not sufficient if gold is not a substantive subject.

## 2. INDIRECT
Use `indirect` only when the article does not primarily concern gold, but contains concrete evidence of a credible connection to the gold market.

Potentially qualifying topics include:
- Monetary policy, interest rates, inflation, central-bank policy, exchange rates, or major currency movements.
- International financial markets, financial crises, sovereign risk, or systemic financial instability.
- Geopolitical conflicts, sanctions, or international events with a clearly supported connection to gold markets.

The topic category alone is not sufficient.

The article must contain at least one of the following:
1. An explicit statement connecting the event to gold prices, gold demand, gold trading, or gold-market conditions.
2. A concrete financial or market mechanism supported by the article, with enough information to establish a credible connection to gold.
3. Specific market evidence, such as a reported movement in gold prices associated with the event.

Do not classify an article as `indirect` solely because it concerns politics, disasters, oil prices, crime, elections, public safety, environmental issues, or general economic activity.

Do not assume that uncertainty, public concern, or investor anxiety automatically increases demand for gold.

Do not invent a relationship through phrases such as "gold is a safe haven" unless the article provides evidence that makes this relationship relevant to the specific event.

## 3. NOT_RELEVANT
Use `not_relevant` when the article has no substantive, evidence-supported connection to gold markets.

Examples:
- Local crime, celebrity news, religious ceremonies, routine government activities, or general social news without a supported gold-market connection.
- Natural disasters or weather warnings without evidence connecting them to gold-market conditions.
- Oil or energy price news without a supported connection to gold prices, financial markets, inflation, or monetary conditions in the article.
- General political disputes or policy announcements without a supported financial or gold-market mechanism.

Do not create a connection merely because a hypothetical chain of economic consequences can be imagined.

## 4. UNCERTAIN
Use `uncertain` only when the article cannot be reliably classified because the available text is insufficient, ambiguous, contradictory, or incomplete.

Examples:
- The title suggests a gold-market topic, but the article body does not provide enough information.
- The article refers to a financial indicator without sufficient context to determine its relationship to gold.
- The source text is corrupted or essential information is missing.

Do not use `uncertain` as a default for ordinary articles. If the article is clearly unrelated to gold, choose `not_relevant`.

# EVIDENCE RULES
1. Use only the supplied title and article text.
2. Never use outside knowledge to fill missing evidence.
3. Never infer that an event affected gold prices unless the article supports that claim.
4. Distinguish an event that occurred from an actual or explicitly described market impact.
5. A plausible theoretical relationship is not sufficient for `indirect`.
6. Prefer `not_relevant` over speculative `indirect` classifications.
7. Do not confuse technical classification failures with semantic uncertainty. If classification cannot be completed because of a processing error, return a technical error rather than inventing a label.
8. Be conservative. False-positive `indirect` classifications can contaminate a financial-news RAG system.

# DECISION PROCEDURE
Step 1: Identify the article's primary subject.
Step 2: Determine whether gold or the gold market is a substantive subject.
Step 3: If yes, check whether the article provides sufficient evidence for `direct`.
Step 4: If not, look for an explicit or concretely supported connection to gold markets.
Step 5: If a credible connection is supported by the article, choose `indirect`.
Step 6: If the connection is speculative or absent, choose `not_relevant`.
Step 7: Use `uncertain` only when the available article text prevents a reliable decision.

# OUTPUT FORMAT
Return exactly one valid JSON object. Do not include Markdown fences or additional commentary.

Required fields:
- `relevance`: one of `direct`, `indirect`, `not_relevant`, `uncertain`
- `confidence`: integer from 0 to 100
- `reason`: concise explanation in Thai
- `evidence`: array of exact quotes copied from the supplied article text
- `market_link`: concise explanation of the supported connection to gold markets, or an empty string if no connection is established
- `review_required`: boolean

# OUTPUT CONSTRAINTS
- For `direct`, evidence must identify the article text explicitly discussing gold or the gold market.
- For `indirect`, evidence must support the actual connection or the concrete mechanism linking the event to gold markets. Do not provide only the event description.
- For `not_relevant`, evidence may be an empty array if no relevant evidence exists. Explain briefly why the article lacks a supported gold-market connection.
- For `uncertain`, identify the missing or ambiguous information.
- Never fabricate quotations. Every evidence item must be an exact substring of the supplied title or article text.
- If evidence is insufficient for `direct` or `indirect`, do not assign either label.
- Set `review_required` to true when confidence is below 80, the evidence is ambiguous, or the article contains a potentially important but unverified market connection.
- Set `review_required` to false only when the classification is well supported by the article.
""".strip()


HUMAN_TEMPLATE = """
หัวข้อข่าว:
{title}

วันที่เผยแพร่:
{published_at}

บทคัดย่อ:
{abstract}

เนื้อหาข่าว:
{content}

โปรดตรวจสอบความเกี่ยวข้องกับตลาดทองคำตามเกณฑ์ที่กำหนด
และตอบกลับเป็น JSON object เท่านั้น
""".strip()


# --------------------------------------------------------
# 9. UNIT TESTS (mock output เท่านั้น ไม่เรียก API)
# --------------------------------------------------------
_AUDIT_OK_MOCK = {
    "relevance": "direct",
    "confidence": 92,
    "reason": "ข่าวรายงานราคาทองคำรายวัน",
    "evidence": ["ราคาทองคำ ลบ 650 บาท"],
    "market_link": "ราคาทองคำแท่งในประเทศ",
    "review_required": False,
}


def _run_parser_tests():
    results = []

    def check(name, fn):
        try:
            fn()
            results.append((name, "PASS", ""))
        except AssertionError as exc:
            results.append((name, "FAIL", str(exc)))
        except Exception as exc:
            results.append((name, "ERROR", f"{type(exc).__name__}: {exc}"))

    def expect_kind(kind, raw):
        try:
            parse_audit_json(raw)
        except AuditFormatError as exc:
            assert exc.kind == kind, f"expected {kind}, got {exc.kind}"
            return
        except Exception as exc:
            raise AssertionError(f"expected AuditFormatError, got {type(exc).__name__}") from exc
        raise AssertionError(f"expected AuditFormatError({kind}), but parse succeeded")

    # 1. JSON object ตรง ๆ
    def t1():
        out = parse_audit_json(json.dumps(_AUDIT_OK_MOCK, ensure_ascii=False))
        assert out["relevance"] == "direct", out
        assert out["confidence"] == 92, out
        assert out["status"] == "ok", out

    # 2. Markdown code fence
    def t2():
        raw = "```json\n" + json.dumps(_AUDIT_OK_MOCK, ensure_ascii=False) + "\n```"
        assert parse_audit_json(raw)["relevance"] == "direct"

    # 3. JSON หลัง <think>...</think>
    def t3():
        raw = "<think>\nข่าวนี้พูดถึงทองคำโดยตรง\n</think>\n" + json.dumps(_AUDIT_OK_MOCK, ensure_ascii=False)
        assert parse_audit_json(raw)["relevance"] == "direct"

    # 4. ข้อความประกอบก่อน JSON
    def t4():
        raw = "นี่คือผลการตรวจสอบ:\n" + json.dumps(_AUDIT_OK_MOCK, ensure_ascii=False) + "\nจบ"
        assert parse_audit_json(raw)["relevance"] == "direct"

    # 5. JSON ถูกตัดกลางข้อความ
    def t5():
        raw = '{"relevance": "direct", "confidence": 90, "reason": "ถูกตัด'
        expect_kind("truncated", raw)

    # 6. ตัวอย่างใน reasoning แล้วคำตอบจริงถูกตัด -> ห้ามเลือกตัวอย่าง
    def t6():
        raw = (
            "<think>ตัวอย่างผลลัพธ์: "
            + json.dumps(_AUDIT_OK_MOCK, ensure_ascii=False)
            + "\n</think>\nคำตอบ: {\"relevance\": \"indirect\", \"confidence\": 80,"
        )
        expect_kind("truncated", raw)

    # 7. JSON หลาย object -> เลือกตัวสุดท้ายที่มี relevance
    def t7():
        first = dict(_AUDIT_OK_MOCK)
        second = dict(_AUDIT_OK_MOCK, relevance="indirect", confidence=70)
        raw = json.dumps(first, ensure_ascii=False) + "\n" + json.dumps(second, ensure_ascii=False)
        out = parse_audit_json(raw)
        assert out["relevance"] == "indirect", out
        assert out["confidence"] == 70, out

    # 8. { } และ escaped quote ภายใน string
    def t8():
        mock = dict(_AUDIT_OK_MOCK, reason='ราคาทอง "{ขึ้น}" 2%', evidence=['เขากล่าวว่า "ทอง" น่าสนใจ'])
        raw = json.dumps(mock, ensure_ascii=False)
        out = parse_audit_json(raw)
        assert out["reason"] == 'ราคาทอง "{ขึ้น}" 2%', out
        assert out["evidence"] == ['เขากล่าวว่า "ทอง" น่าสนใจ'], out

    # 9. missing fields
    def t9():
        expect_kind("schema_error", '{"relevance": "direct"}')

    # 10. invalid label
    def t10():
        mock = dict(_AUDIT_OK_MOCK, relevance="gold_news")
        expect_kind("schema_error", json.dumps(mock, ensure_ascii=False))

    # 11. ชนิดข้อมูลผิด schema
    def t11():
        expect_kind("schema_error", json.dumps(
            dict(_AUDIT_OK_MOCK, evidence="not a list"), ensure_ascii=False))
        expect_kind("schema_error", json.dumps(
            dict(_AUDIT_OK_MOCK, review_required="true"), ensure_ascii=False))
        expect_kind("schema_error", json.dumps(
            dict(_AUDIT_OK_MOCK, confidence="สูงมาก"), ensure_ascii=False))
        expect_kind("schema_error", json.dumps(
            dict(_AUDIT_OK_MOCK, confidence=150), ensure_ascii=False))

    # 12. confidence เป็น float ที่แปลงได้
    def t12():
        mock = dict(_AUDIT_OK_MOCK, confidence=88.0)
        assert parse_audit_json(json.dumps(mock, ensure_ascii=False))["confidence"] == 88

    # 13. output ว่าง / ไม่มี JSON
    def t13():
        expect_kind("parse_error", "")
        expect_kind("parse_error", "ไม่พบข้อมูลที่เกี่ยวข้อง")
        expect_kind("truncated", "<think>กำลังวิเคราะห์ข่าวนี้")

    # 14. technical failure ห้ามกลายเป็น uncertain อัตโนมัติ
    def t14():
        try:
            parse_audit_json("<think>วิเคราะห์")
        except AuditFormatError as exc:
            assert exc.kind in TECHNICAL_ERROR_TYPES
            assert exc.kind != "uncertain"
            return
        raise AssertionError("expected AuditFormatError")

    # 15. resume: ข้าม record ที่สำเร็จแล้ว (ไม่มีการเรียก API)
    def t15():
        ok = {"article_id": "a1", "audit": dict(_AUDIT_OK_MOCK, status="ok")}
        failed = {"article_id": "a2", "audit": {
            "relevance": None, "confidence": None, "reason": "technical",
            "evidence": [], "market_link": "", "review_required": True,
            "status": "failed", "error_type": "truncated", "error": "x",
        }}
        state = {"a1": ok, "a2": failed}
        assert find_pending_ids([ok, failed], state) == ["a2"]
        assert find_pending_ids([ok], {"a1": ok}) == []

    # 16. dedupe: article_id ซ้ำ -> last wins
    def t16():
        a = {"article_id": "a1", "n": 1}
        b = {"article_id": "a1", "n": 2}
        assert dedupe_by_article_id([a, b])["a1"]["n"] == 2
        assert find_pending_ids([a, a, b], {}) == ["a1"]

    # 17. scrub secrets
    def t17():
        assert THAILLM_API_KEYS[0] not in scrub_secrets(f"err {THAILLM_API_KEYS[0]} end")
        assert "sk-" not in scrub_secrets("Authorization: Bearer sk-abcdef1234567890")

    for name, fn in [
        ("1 direct JSON", t1),
        ("2 markdown fence", t2),
        ("3 after <think>", t3),
        ("4 preamble text", t4),
        ("5 truncated JSON", t5),
        ("6 truncated answer after example", t6),
        ("7 multiple objects -> last with relevance", t7),
        ("8 braces/escaped quotes in string", t8),
        ("9 missing fields", t9),
        ("10 invalid label", t10),
        ("11 schema type errors", t11),
        ("12 float confidence", t12),
        ("13 empty / no JSON", t13),
        ("14 failure != uncertain", t14),
        ("15 resume skips completed", t15),
        ("16 dedupe by article_id", t16),
        ("17 scrub secrets", t17),
    ]:
        check(name, fn)

    return results


_parser_test_results = _run_parser_tests()

_passed = sum(1 for _, s, _ in _parser_test_results if s == "PASS")

for _name, _status, _detail in _parser_test_results:
    print(f"[{_status:5s}] {_name}{(' | ' + _detail) if _detail else ''}")

print(f"\nParser tests: {_passed}/{len(_parser_test_results)} passed")

assert _passed == len(_parser_test_results), "parser unit tests failed — ห้ามรัน retry จริง"


[PASS ] 1 direct JSON
[PASS ] 2 markdown fence
[PASS ] 3 after <think>
[PASS ] 4 preamble text
[PASS ] 5 truncated JSON
[PASS ] 6 truncated answer after example
[PASS ] 7 multiple objects -> last with relevance
[PASS ] 8 braces/escaped quotes in string
[PASS ] 9 missing fields
[PASS ] 10 invalid label
[PASS ] 11 schema type errors
[PASS ] 12 float confidence
[PASS ] 13 empty / no JSON
[PASS ] 14 failure != uncertain
[PASS ] 15 resume skips completed
[PASS ] 16 dedupe by article_id
[PASS ] 17 scrub secrets

Parser tests: 17/17 passed


In [6]:
# ========================================================
# CELL 8: RETRY เฉพาะ TECHNICAL FAILURES -> gold_relevance_audit_v2.jsonl
# - retry เฉพาะ record ที่ยังไม่มีผล audit ที่ผ่าน schema
# - ไม่แตะผลที่สำเร็จแล้ว และไม่เขียนทับไฟล์ต้นฉบับ
# - บันทึกผลทันทีต่อ record + atomic write ทุกครั้งที่เขียนทับ
# ========================================================

import time
import hashlib
from collections import Counter
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from threading import Lock

from tqdm import tqdm
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


# --------------------------------------------------------
# 1. Auditor LLM
#    endpoint ThaiLLM (OpenAI-compatible) ไม่ได้ประกาศรองรับ native JSON mode
#    จึงใช้ strict prompt + balanced-brace parser + schema validation แทน
# --------------------------------------------------------
MAX_ATTEMPTS = 3        # retry สูงสุด 3 ครั้งต่อ record (ชั้น app)
MAX_TOKENS = 1500       # เดิม 800 ไม่พอสำหรับ JSON + reason ภาษาไทย

audit_llms = [
    ChatOpenAI(
        model=THAILLM_MODEL,
        base_url=THAILLM_BASE_URL,
        api_key=api_key,
        temperature=0,
        max_tokens=MAX_TOKENS,
        timeout=90,
        max_retries=0,      # ให้ loop ของเราเป็นผู้คุมจำนวนครั้งทั้งหมด
    )
    for api_key in THAILLM_API_KEYS
]
num_keys = len(audit_llms)

audit_prompt = ChatPromptTemplate.from_messages([
    ("system", AUDITOR_SYSTEM_PROMPT),
    ("human", HUMAN_TEMPLATE),
])
audit_chains = [audit_prompt | llm | StrOutputParser() for llm in audit_llms]

print(f"Auditor instances: {num_keys} | max_tokens={MAX_TOKENS} | max_attempts={MAX_ATTEMPTS}")


def get_audit_index(article_id):
    """กระจายข่าวลง LLM แต่ละตัวด้วย hash ของ article_id (mod N)"""
    hash_value = int(hashlib.md5(article_id.encode()).hexdigest(), 16)
    return hash_value % max(1, num_keys)


def audit_article(article, llm_index):
    """ตรวจ 1 ข่าว — คืน (result, failure)

    แยก api_error / timeout / rate_limit / truncated / parse_error / schema_error
    และ retry ไม่เกิน MAX_ATTEMPTS ครั้ง
    """
    title = str(article.get("title") or "")
    abstract = str(article.get("abstract") or "")
    content = str(article.get("content") or "")

    if not title.strip() and not content.strip():
        return None, {
            "error_type": "parse_error",
            "error": "empty title and content",
            "attempts": 0,
        }

    chain = audit_chains[llm_index]
    last_failure = None

    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            response = chain.invoke({
                "title": title,
                "published_at": article.get("published_at") or "ไม่ทราบ",
                "abstract": abstract[:2000],
                "content": content[:7000],
            })

        except Exception as exc:
            error_type = classify_api_error(exc)
            detail = scrub_secrets(exc)
            last_failure = {
                "error_type": error_type,
                "error": detail[:500],
                "attempts": attempt,
            }
            print(
                f"[AUDIT-{llm_index+1}] {article.get('article_id')} "
                f"attempt {attempt}/{MAX_ATTEMPTS} -> {error_type}: {detail[:150]}"
            )

        else:
            try:
                return parse_audit_json(response), None

            except AuditFormatError as exc:
                detail = scrub_secrets(str(exc))
                last_failure = {
                    "error_type": exc.kind,
                    "error": detail[:500],
                    "preview": scrub_secrets(exc.preview)[:300],
                    "attempts": attempt,
                }
                print(
                    f"[AUDIT-{llm_index+1}] {article.get('article_id')} "
                    f"attempt {attempt}/{MAX_ATTEMPTS} -> {exc.kind}: {detail[:150]}"
                )

        if attempt < MAX_ATTEMPTS:
            time.sleep(min(2 ** attempt, 15))

    return None, last_failure


def build_audit_record(item, llm_index, result, failure):
    """ประกอบ record สำหรับ v2 (สำเร็จ = ผลจริง, ล้มเหลว = คงสถานะ technical failure)"""
    if result is not None:
        audit = result
    else:
        # ห้ามปลอม label: technical failure ต้องไม่กลายเป็น direct/indirect/not_relevant/uncertain
        audit = {
            "relevance": None,
            "confidence": None,
            "reason": "Technical failure: audit did not produce a schema-valid result",
            "evidence": [],
            "market_link": "",
            "review_required": True,
            "status": "failed",
            "error_type": (failure or {}).get("error_type"),
            "error": (failure or {}).get("error"),
            "error_preview": (failure or {}).get("preview", ""),
            "attempts": (failure or {}).get("attempts"),
        }

    record = dict(item)
    record["audit"] = audit
    record["audit_metadata"] = {
        "provider": "ThaiLLM",
        "model": THAILLM_MODEL,
        "framework": "LangChain",
        "audited_at": datetime.now(timezone.utc).isoformat(),
        "prompt_version": "gold-relevance-audit-v2",
        "llm_instance": llm_index + 1,
        "max_tokens": MAX_TOKENS,
        "max_attempts": MAX_ATTEMPTS,
    }
    return record


# --------------------------------------------------------
# 2. เลือกเฉพาะ record ที่ล้มเหลวทางเทคนิค
# --------------------------------------------------------
RETRY_LIMIT = 3     # ทดลอง 3 records ก่อน; ตั้งเป็น None เมื่อต้องการ retry ที่เหลือทั้งหมด

all_records = read_jsonl_file(AUDIT_V1_FILE)
assert all_records, f"ไม่พบข้อมูลใน {AUDIT_V1_FILE}"

article_by_id = dedupe_by_article_id(all_records)

# state = ผลที่สำเร็จแล้ว (จาก v2) + seed จาก v1 — ไม่ลบและไม่เขียนทับผลเดิม
state, seeded = load_v2_state()

pending_ids = find_pending_ids(all_records, state)
selected_ids = pending_ids if RETRY_LIMIT is None else pending_ids[:RETRY_LIMIT]
batch = [article_by_id[article_id] for article_id in selected_ids]


def ordered_records():
    """เรียง record ตามลำดับเดิมของไฟล์ v1 เพื่อผลลัพธ์ที่เทียบกันได้"""
    ordered = [state[article_id] for article_id in article_by_id if article_id in state]
    ordered += [record for article_id, record in state.items() if article_id not in article_by_id]
    return ordered


# เขียน v2 ครั้งแรก: seed ผลที่สำเร็จแล้ว (atomic write, กัน record ซ้ำ)
write_jsonl_atomic(AUDIT_V2_FILE, ordered_records())

v1_ok = sum(1 for r in article_by_id.values() if normalize_v1_audit(r.get("audit")))

print(f"\nv1 records                : {len(all_records)}")
print(f"v1 valid results          : {v1_ok}")
print(f"seeded into v2            : {seeded}")
print(f"technical failures (v1)   : {len(pending_ids)}")
print(f"records selected this run : {len(batch)}")
print(f"v2 file                   : {AUDIT_V2_FILE}")


# --------------------------------------------------------
# 3. Retry (parallel, บันทึกผลต่อ record ทันที)
# --------------------------------------------------------
write_lock = Lock()
repaired = 0
still_failing = 0


def process_article(item):
    llm_index = get_audit_index(item["article_id"])
    result, failure = audit_article(item, llm_index)
    return build_audit_record(item, llm_index, result, failure)


if batch:
    with ThreadPoolExecutor(max_workers=max(1, num_keys)) as executor:
        future_to_item = {
            executor.submit(process_article, item): item
            for item in batch
        }

        with tqdm(total=len(batch), desc="Repairing", unit="article") as pbar:
            for future in as_completed(future_to_item):
                item = future_to_item[future]
                article_id = item.get("article_id")

                try:
                    record = future.result()
                except Exception as exc:
                    print(f"worker error {article_id}: {scrub_secrets(exc)[:150]}")
                    still_failing += 1
                    pbar.update(1)
                    continue

                with write_lock:
                    state[article_id] = record
                    append_jsonl(AUDIT_V2_FILE, record)

                    if not is_audit_ok(record):
                        append_jsonl(AUDIT_RETRY_LOG, {
                            "article_id": article_id,
                            "error_type": (record.get("audit") or {}).get("error_type"),
                            "error": (record.get("audit") or {}).get("error"),
                            "attempts": (record.get("audit") or {}).get("attempts"),
                            "logged_at": datetime.now(timezone.utc).isoformat(),
                        })

                if is_audit_ok(record):
                    repaired += 1
                else:
                    still_failing += 1

                pbar.set_postfix({"repaired": repaired, "failed": still_failing})
                pbar.update(1)

                time.sleep(THAILLM_DELAY)

# เขียนทับ v2 แบบ atomic: dedupe ตาม article_id (กัน duplicate เมื่อ rerun)
write_jsonl_atomic(AUDIT_V2_FILE, ordered_records())


# --------------------------------------------------------
# 4. Audit split (เฉพาะผลที่ผ่าน schema — ไม่รวม technical failure)
# --------------------------------------------------------
buckets = {"direct": [], "indirect": [], "uncertain": []}
failed_records = []

for record in state.values():
    if is_audit_ok(record):
        label = record["audit"]["relevance"]
        if label in buckets:
            buckets[label].append(record)
    else:
        failed_records.append(record)

for label in ("direct", "indirect", "uncertain"):
    write_jsonl_atomic(OUT_DIR / f"gold_audit_{label}.jsonl", buckets[label])


# --------------------------------------------------------
# 5. Report
# --------------------------------------------------------
v2_records = read_jsonl_file(AUDIT_V2_FILE)
v2_ids = dedupe_by_article_id(v2_records)
duplicate_lines = len(v2_records) - len(v2_ids)

remaining_ids = find_pending_ids(all_records, dedupe_by_article_id(v2_records))

v2_labels = Counter(
    record["audit"]["relevance"]
    for record in state.values()
    if is_audit_ok(record)
)
error_types = Counter(
    (record.get("audit") or {}).get("error_type")
    for record in failed_records
)

print("\n" + "=" * 60)
print("AUDIT RETRY REPORT (v2)")
print("=" * 60)
print(f"records eligible for retry : {len(pending_ids)}")
print(f"records retried this run   : {len(batch)}")
print(f"repaired this run          : {repaired}")
print(f"still failing this run     : {still_failing}")
print(f"remaining to retry         : {len(remaining_ids)}")
print("-" * 60)
print(f"v2 total records           : {len(v2_records)}")
print(f"v2 schema-valid results    : {sum(1 for r in state.values() if is_audit_ok(r))}")
print(f"v2 technical failures      : {len(failed_records)}")
print(f"duplicate lines in v2      : {duplicate_lines}")
print("-" * 60)
print("v2 labels (schema-valid only):")
for label, count in v2_labels.most_common():
    print(f"  {label:15s} {count:5,}")
print("-" * 60)
print("failure types:")
for error_type, count in error_types.most_common():
    print(f"  {str(error_type):15s} {count:5,}")
print("-" * 60)
print("split files (failures excluded):")
for label in ("direct", "indirect", "uncertain"):
    print(f"  {label:15s} {len(buckets[label]):5,}  -> gold_audit_{label}.jsonl")
print("-" * 60)
print("v1 file (unchanged) :", AUDIT_V1_FILE)
print("v2 file             :", AUDIT_V2_FILE)
print("retry log           :", AUDIT_RETRY_LOG)
print("=" * 60)

if remaining_ids and RETRY_LIMIT is not None:
    print(
        f"\nยังเหลือ {len(remaining_ids)} records ที่ต้อง retry "
        "— ตั้ง RETRY_LIMIT = None แล้วรัน CELL 8 ซ้ำเพื่อ retry ที่เหลือ "
        "(records ที่สำเร็จแล้วจะไม่ถูกเรียก API ซ้ำ)"
    )


Auditor instances: 7 | max_tokens=1500 | max_attempts=3

v1 records                : 928
v1 valid results          : 879
seeded into v2            : 0
technical failures (v1)   : 46
records selected this run : 3
v2 file                   : D:\Project_Github\gold assisstant\dataset\news\gold_relevance_audit_v2.jsonl


Repairing: 100%|██████████| 3/3 [00:04<00:00,  1.45s/article, repaired=3, failed=0]



AUDIT RETRY REPORT (v2)
records eligible for retry : 46
records retried this run   : 3
repaired this run          : 3
still failing this run     : 0
remaining to retry         : 43
------------------------------------------------------------
v2 total records           : 885
v2 schema-valid results    : 885
v2 technical failures      : 0
duplicate lines in v2      : 0
------------------------------------------------------------
v2 labels (schema-valid only):
  not_relevant      844
  indirect           20
  uncertain          12
  direct              9
------------------------------------------------------------
failure types:
------------------------------------------------------------
split files (failures excluded):
  direct              9  -> gold_audit_direct.jsonl
  indirect           20  -> gold_audit_indirect.jsonl
  uncertain          12  -> gold_audit_uncertain.jsonl
------------------------------------------------------------
v1 file (unchanged) : D:\Project_Github\gold assi